In [ ]:
# 必要なライブラリのインポート
import os
import time
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from pyscf import gto
from gpu4pyscf.dft import rks
from pyscf.tools import cubegen, molden

# --- 計算設定 ---
XC_FUNCTIONAL = 'b3lyp'          # 汎関数
BASIS_SET = '6-31+g**'           # 基底関数 (6-31+G(d,p))
SMILES = 'C=1(C=2C(C=CC1)=CC=CC2)N(C=3C=CC=CC3)C=4C=CC(C=5C=CC(N(C=6C=7C(C=CC6)=CC=CC7)C=8C=CC=CC8)=CC5)=CC4' # NPDのSMILES
MOL_NAME = 'NPD_RESP'            # 分子名（出力ファイル名などに使用）

# 出力ディレクトリ
out_dir = f"output/{MOL_NAME}"
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# 1. RDKitによる初期構造生成とMMFF最適化
print(f"[{MOL_NAME}] RDKitで3D構造を生成中...")
mol_rdkit = Chem.MolFromSmiles(SMILES)
mol_rdkit = Chem.AddHs(mol_rdkit)
AllChem.EmbedMolecule(mol_rdkit, randomSeed=42)
AllChem.MMFFOptimizeMolecule(mol_rdkit)

# RDKitの構造を一時XYZファイルとして保存
rdkit_xyz = os.path.join(out_dir, f"{MOL_NAME}_rdkit.xyz")
Chem.MolToXYZFile(mol_rdkit, rdkit_xyz)
print(f"RDKit構造を保存しました: {rdkit_xyz}")


In [ ]:
# 2. GPU4PySCFによる構造最適化と単点計算 (RESP用)
from pyscf.geomopt.geometric_solver import optimize

print(f"[{MOL_NAME}] PySCF (GPU) で計算を開始します (Level: {XC_FUNCTIONAL}/{BASIS_SET})")
mol = gto.M(atom=rdkit_xyz, basis=BASIS_SET, charge=0, spin=0)
mol.output = os.path.join(out_dir, f"{MOL_NAME}.log")
mol.verbose = 4
mol.build()

mf = rks.RKS(mol, xc=XC_FUNCTIONAL)

# 構造最適化
xyz_path = os.path.join(out_dir, f"{MOL_NAME}_opt.xyz")
if os.path.exists(xyz_path):
    print("-> 既存の最適化構造が見つかりました。最適化をスキップします。")
    mol_opt = gto.M(atom=xyz_path, basis=BASIS_SET, charge=0, spin=0)
    mol_opt.output = os.path.join(out_dir, f"{MOL_NAME}.log")
    mol_opt.verbose = 4
else:
    print("-> 構造最適化を実行中...")
    start_time = time.time()
    mol_opt = optimize(mf, maxsteps=100)
    print(f"構造最適化 完了 (所要時間: {time.time() - start_time:.1f}秒)")
    
    # 座標の保存
    opt_coords = mol_opt.atom_coords()
    opt_symbols = [mol_opt.atom_symbol(i) for i in range(mol_opt.natm)]
    xyz_lines = [f"{mol_opt.natm}", f"{MOL_NAME} optimized"]
    for sym, pos in zip(opt_symbols, opt_coords):
        pos_ang = pos * 0.529177210903
        xyz_lines.append(f"{sym} {pos_ang[0]:.5f} {pos_ang[1]:.5f} {pos_ang[2]:.5f}")
    with open(xyz_path, "w") as f_xyz:
        f_xyz.write("\n".join(xyz_lines) + "\n")

# 最適化構造での単点計算 (chkファイル出力用)
print("-> 電子状態の単点計算を実行中...")
mf_opt = rks.RKS(mol_opt, xc=XC_FUNCTIONAL).density_fit()
chk_path = os.path.join(out_dir, f"{MOL_NAME}.chk")
mf_opt.chkfile = chk_path
mf_opt.grids.level = 1
mf_opt.kernel()

# 3. CPUに転送して電荷とCubeファイル、Moldenの出力
print("-> 結果の解析とファイル出力中...")
mf_cpu = mf_opt.to_cpu()
pop, chg = mf_cpu.mulliken_pop()

# Moldenファイルの出力 (MultiwfnでRESP電荷計算に利用)
molden_path = os.path.join(out_dir, f"{MOL_NAME}.molden")
molden.from_chkfile(molden_path, chk_path)
print(f"Moldenファイルを出力しました: {molden_path}")

# ESP (MEP) Cubeファイルの出力
mep_path = os.path.join(out_dir, f"{MOL_NAME}_MEP.cube")
cubegen.mep(mol_opt, mep_path, mf_cpu.make_rdm1())
print(f"ESP Cubeファイルを出力しました: {mep_path}")

print("全ての処理が完了しました！")
